# CCI V2.1-D1 — Classical Benchmark (Kaggle execution)

Re-run of the D1 classical benchmark under the **revised V2.1 protocol**
(ADR-011). The first D1 execution completed operationally but returned a
degenerate null: every candidate collapsed to the frozen S7 fallback,
because the old inner calibration window (`train` 2024-05-01 to
2024-06-30) sat inside S7's own fit scope and was therefore in-sample for
the fallback.

**What changed, and only this:** the temporal placement of the three
windows. Fit is now `train` 2023-08-01 to 2024-06-30, calibration is
`validation` 2024-07-01 to 2024-09-30, and outer evaluation is
`validation` 2024-10-01 to 2024-12-31. The candidate catalog, the
estimator settings, the scientific gates, the safety margins, the
architecture, and the batch size are unchanged, so V2.1 against V2.0 is a
controlled comparison.

The protocol now validates that no calibration or evaluation window
intersects the frozen fallback's fit scope, and the runner publishes
Stage-A override counts, a fallback-only baseline, and a relative
improvement delta per candidate. A candidate that never overrides, or
that fails to beat the fallback baseline, is not selectable.

**Boundary:** the bundle contains code, frozen configs, and the S7
fallback package. The only data file is the development-only
`scientific.parquet` (train + validation). `test`, `stress`, and
`monitor` remain sealed and have no unlock path in this code.

All computational logic lives in the shipped package
(`consumer_complaint_intelligence.kaggle_execution`); cells only
orchestrate and print aggregate evidence.

In [ ]:
%pip install --quiet scikit-learn==1.9.0 imbalanced-learn==0.14.2

In [ ]:
import sys
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/temp/project")
OUTPUT_ROOT = Path("/kaggle/working")


def _input_listing(limit=200):
    return [str(path) for path in sorted(INPUT_ROOT.rglob("*"))[:limit]]


manifests = [
    path
    for path in sorted(INPUT_ROOT.rglob("kaggle_bundle_manifest.json"))
    if (path.parent / "src").is_dir()
]
if manifests:
    bundle_root = manifests[0].parent
else:
    zips = sorted(INPUT_ROOT.rglob("cci-v2-bundle.zip"))
    if not zips:
        raise FileNotFoundError(
            f"No bundle manifest or zip under {INPUT_ROOT}; "
            f"mounted: {_input_listing()}"
        )
    bundle_root = Path("/kaggle/temp/bundle_extracted")
    bundle_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall(bundle_root)

caches = sorted(INPUT_ROOT.rglob("scientific.parquet"))
if not caches:
    raise FileNotFoundError(
        f"No scientific.parquet under {INPUT_ROOT}; "
        f"mounted: {_input_listing()}"
    )
CACHE_FILE = caches[0]
print("bundle_root:", bundle_root)
print("cache_file:", CACHE_FILE)

sys.path.insert(0, str(bundle_root / "src"))
from consumer_complaint_intelligence import kaggle_execution as kx

kx.assert_pinned_environment()

In [ ]:
staging = kx.stage_project(bundle_root, CACHE_FILE, WORK_ROOT)
print(staging)
print(kx.preflight(WORK_ROOT))

In [ ]:
result = kx.run_full(WORK_ROOT)
print(
    {
        "status": result["status"],
        "complete": result["complete"],
        "runtime_seconds": result["runtime_seconds"],
        "candidate_count": len(result["candidates"]),
        "selected": result["selected"],
    }
)

In [ ]:
print(kx.collect_outputs(WORK_ROOT, OUTPUT_ROOT))

## Retrieval

The staged tree lives under `/kaggle/temp` and is discarded with the
session. Only the two aggregate JSON files are persisted as notebook
output:

- `v2_classical_benchmark.json` → local `temp/v2/`
- `v2_classical_results.json` → local `config/`

Download them (`kaggle kernels output`) and validate locally with
`validate_v2_manifest` plus the `tests/test_v2_*` suite before any
selection decision.